<a href="https://colab.research.google.com/github/SriSharanya-617/deeplearning/blob/main/CarPrice_Prediction_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
path='/content/drive/MyDrive/Colab Notebooks/CarPrice_dataset.csv'
df=pd.read_csv(path)
df.head()


Mounted at /content/drive


,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [2]:
df.isnull().sum()

,0
car_ID,0
symboling,0
CarName,0
fueltype,0
aspiration,0
doornumber,0
carbody,0
drivewheel,0
enginelocation,0
wheelbase,0


In [4]:
from pandas.core.arrays import categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

X=df.drop(columns='price')
y=df['price']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist ()

#Define Preprocessing steps
preprocessor = ColumnTransformer(
  transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('car', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
  ]
)

#Apply Preprocessing
X_processed=preprocessor.fit_transform(X)

#Split the data
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

#Define the model
model=Sequential([
  Dense(512, activation='relu', input_shape=(X_train.shape[1], )),
  Dense(256, activation='relu'),
  Dense(128, activation='relu'),
  Dense(64, activation='relu'),
  Dense(32, activation='relu'),
  Dense(1) # Output layer for regression
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(optimizer='adam', loss='mean_absolute_error',
  metrics=['mean_absolute_error','mean_squared_error'])

In [9]:
history=model.fit(X_train,y_train,epochs=70,validation_split=0.2,verbose=1)

Epoch 1/70
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 496ms/step - loss: 13041.3613 - mean_absolute_error: 13041.3613 - mean_squared_error: 219984528.0000 - val_loss: 15080.4219 - val_mean_absolute_error: 15080.4219 - val_mean_squared_error: 317856544.0000
Epoch 2/70
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 12758.0029 - mean_absolute_error: 12758.0029 - mean_squared_error: 218640704.0000 - val_loss: 15073.1289 - val_mean_absolute_error: 15073.1289 - val_mean_squared_error: 317621504.0000
Epoch 3/70
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 12874.8965 - mean_absolute_error: 12874.8965 - mean_squared_error: 219901392.0000 - val_loss: 15052.5869 - val_mean_absolute_error: 15052.5869 - val_mean_squared_error: 316969856.0000
Epoch 4/70
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 12555.1162 - mean_absolute_error: 12555.1162 - mean_squared_error: 209583248.0000 - val_loss: 15000.5205 - val_mean_absolute_error: 15000.5205 - val_mean_squared_error: 315352608.0000
Epoch 5/70
5/5 ━━━━━━━━━━━━━━━━━━━━

In [11]:
#Model Evalution
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
y_pred=model.predict(X_test)
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
r2=r2_score(y_test,y_pred)

print(f'Mean Absolute Error: {mae}')
print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Mean Absolute Error: 1925.966094035823
Mean Squared Error: 9264723.284153717
R-squared: 0.8826418621585431


In [13]:
# Example New Data
new_data=X.iloc[[0]]
new_data_precessed=preprocessor.transform(new_data)

predicted_price=model.predict(new_data_precessed)
print(f'Predicted Price: {predicted_price[0][0]}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 391ms/step
Predicted Price: 14224.5810546875
